# 15.3 Arrays, Strings and Two Pointers

**Prerequisites:** 15.1 Complexity Analysis, 15.2 Python's Built-ins  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What an array *is*, and why indexing is O(1)
- **Two pointers** - converging, and fast/slow
- **Sliding window** - fixed size and variable size
- **Prefix sums** - answer range queries in O(1) after O(n) setup
- **In-place** manipulation, and what O(1) extra space really means
- The four patterns that solve a large share of array interview questions
- Worked interview problems, from easy to hard

---

## Arrays, and what Python gives you

An **array** is a contiguous block of memory holding equally sized elements. That layout is the whole story:

```
   address:  1000   1008   1016   1024
             [ 10 ][ 20 ][ 30 ][ 40 ]
               [0]   [1]   [2]   [3]

   data[2]  ->  base + 2 x size  ->  one arithmetic step, O(1)
```

Because the position is *computed*, not searched for, indexing costs the same whether the array has ten elements or ten million.

| | Python `list` | true array (e.g. C, `array`, NumPy) |
|---|---|---|
| Holds | pointers to objects | the values themselves |
| Types | mixed | one type |
| Resizing | automatic | fixed |
| Memory | higher | compact |

A Python list is an array **of pointers**, which is why it can hold mixed types and why it uses more memory than it looks like it should. For interview purposes it behaves as an array; the costs are in **15.2**.

---

# Pattern 1: Two pointers, converging

Two indices start at opposite ends and move toward each other.

```
   [ 1, 3, 5, 7, 9, 11 ]     target = 12
     ^              ^
    left          right      sum = 1 + 11 = 12  -> found
```

**When it applies:** the data is **sorted** (or symmetry makes order irrelevant, as in palindromes), and you are looking for a pair.

**Why it works:** at each step you can rule out an entire side. If the sum is too small, no pair using the current `left` can work — every other candidate is smaller still — so advance `left`. That reasoning is what turns O(n²) into O(n).

| | Brute force | Two pointers |
|---|---|---|
| Time | O(n²) | **O(n)** |
| Space | O(1) | O(1) |

In [ ]:
def two_sum_sorted(numbers, target):
    """Find indices of two numbers summing to target. Input MUST be sorted.

    Time O(n), space O(1).
    """
    left, right = 0, len(numbers) - 1
    steps = 0
    while left < right:
        steps += 1
        current = numbers[left] + numbers[right]
        if current == target:
            return (left, right), steps
        if current < target:
            left += 1                 # need more; the smallest must go
        else:
            right -= 1                # need less; the largest must go
    return None, steps


def two_sum_brute(numbers, target):
    steps = 0
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            steps += 1
            if numbers[i] + numbers[j] == target:
                return (i, j), steps
    return None, steps


data = list(range(1, 2001, 2))          # 1, 3, 5, ... 1999
target = data[-1] + data[0]

pair_a, steps_a = two_sum_sorted(data, target)
pair_b, steps_b = two_sum_brute(data, target)
print(f"n = {len(data)}, target = {target}")
print(f"  two pointers : {pair_a}  in {steps_a:>9,} steps")
print(f"  brute force  : {pair_b}  in {steps_b:>9,} steps")

# the worst case for brute force: the pair is in the middle
middle_target = data[499] + data[500]
_, steps_a = two_sum_sorted(data, middle_target)
_, steps_b = two_sum_brute(data, middle_target)
print(f"\ntarget in the middle:")
print(f"  two pointers : {steps_a:>9,} steps")
print(f"  brute force  : {steps_b:>9,} steps   "
      f"{steps_b / steps_a:,.0f}x more")

In [ ]:
def is_palindrome(text):
    """Ignoring case and non-alphanumerics. O(n) time, O(1) extra space."""
    left, right = 0, len(text) - 1
    while left < right:
        while left < right and not text[left].isalnum():
            left += 1
        while left < right and not text[right].isalnum():
            right -= 1
        if text[left].lower() != text[right].lower():
            return False
        left += 1
        right -= 1
    return True


for sample in ["A man, a plan, a canal: Panama", "race a car", "", "ab"]:
    print(f"  {sample!r:<34} {is_palindrome(sample)}")

print("\nThe Pythonic one-liner is clearer but uses O(n) EXTRA space:")
print("    cleaned = [c.lower() for c in text if c.isalnum()]")
print("    return cleaned == cleaned[::-1]")
print("\nIn an interview, write the readable version, then say you can do")
print("it in O(1) space with two pointers - and do it if asked.")

## Two pointers, fast and slow

Both start at the front; one moves faster. Two very different uses:

**1. Read/write pointers — in-place filtering.** The slow pointer marks where the next kept element goes; the fast pointer scans.

```
   remove all 3s from [1,3,2,3,4]

   fast scans ->  1  3  2  3  4
   slow writes    1     2     4      slow ends at 3 = the new length
```

**2. Cycle detection — Floyd's tortoise and hare.** One moves 1 step, the other 2. If there is a cycle they must eventually meet; if there is not, the fast one hits the end. This becomes essential for linked lists (**15.4**).

In [ ]:
def remove_value(data, unwanted):
    """Remove every occurrence in place. Returns the new length.

    O(n) time, O(1) space - the classic read/write two-pointer.
    """
    slow = 0
    for fast in range(len(data)):
        if data[fast] != unwanted:
            data[slow] = data[fast]
            slow += 1
    return slow


values = [1, 3, 2, 3, 4, 3, 5]
length = remove_value(values, 3)
print(f"kept {length} items: {values[:length]}")
print(f"the tail is now junk: {values}")
print("  ^ in-place algorithms often leave garbage past the new length;")

print("    that is why they return it.\n")


def dedupe_sorted(data):
    """Remove duplicates from a SORTED list in place. O(n) time, O(1) space."""
    if not data:
        return 0
    slow = 0
    for fast in range(1, len(data)):
        if data[fast] != data[slow]:
            slow += 1
            data[slow] = data[fast]
    return slow + 1


sorted_values = [1, 1, 2, 2, 2, 3, 4, 4, 5]
length = dedupe_sorted(sorted_values)
print(f"deduped to {length}: {sorted_values[:length]}")
print("\n🔴 This only works because the input is SORTED - duplicates are")
print("   adjacent. On unsorted data you need a set: O(n) time, O(n) space.")

---

# Pattern 2: Sliding window

A contiguous range that moves through the data. Instead of recomputing from scratch for each position, you **add what enters and subtract what leaves**.

```
   sum of every window of size 3:

   [ 2, 1, 5, 1, 3, 2 ]
     └──────┘              sum = 8
        └──────┘           sum = 8 - 2 + 1 = 7      <- O(1), not O(k)
           └──────┘        sum = 7 - 1 + 3 = 9
```

Recomputing each window is O(n·k). Sliding is **O(n)**.

| Variant | Window size | Typical question |
|---|---|---|
| **Fixed** | given, `k` | max sum of any k consecutive |
| **Variable** | grows/shrinks by a condition | longest substring without repeats |

**Recognising it:** the question says *contiguous*, *consecutive*, *substring* or *subarray*, and asks for a maximum, minimum or count.

In [ ]:
def max_window_sum_naive(data, k):
    operations = 0
    best = float("-inf")
    for start in range(len(data) - k + 1):
        total = 0
        for offset in range(k):          # recompute the whole window
            total += data[start + offset]
            operations += 1
        best = max(best, total)
    return best, operations


def max_window_sum(data, k):
    """Slide: add the entering element, subtract the leaving one. O(n)."""
    operations = 0
    window = sum(data[:k])
    operations += k
    best = window
    for i in range(k, len(data)):
        window += data[i] - data[i - k]  # O(1) per position
        operations += 1
        best = max(best, window)
    return best, operations


import random

rng = random.Random(15)
series = [rng.randint(1, 100) for _ in range(2_000)]
K = 50

naive_best, naive_ops = max_window_sum_naive(series, K)
fast_best, fast_ops = max_window_sum(series, K)

print(f"n = {len(series):,}, window k = {K}")
print(f"  same answer : {naive_best == fast_best} ({fast_best})")
print(f"  recompute   : {naive_ops:>8,} operations   O(n*k)")
print(f"  slide       : {fast_ops:>8,} operations   O(n)")
print(f"  ratio       : {naive_ops / fast_ops:.0f}x")
print("\n  The saving grows with k. At k = 500 it would be 10x larger again.")

### Variable-size windows

Harder, and far more common in interviews. The shape is always the same:

```
    left = 0
    for right in range(len(data)):
        ...include data[right] in the window...

        while <the window is invalid>:
            ...remove data[left]...
            left += 1

        ...the window is now valid: record the answer...
```

**Why it is O(n) and not O(n²):** `right` advances n times, and `left` only ever advances — never resets. Each index enters and leaves the window at most once, so there are at most 2n pointer moves regardless of how the `while` nests inside the `for`.

> 🔴 That argument — *amortised over the whole run, each element is processed a constant number of times* — is what interviewers want to hear. A nested `while` inside a `for` looks quadratic until you make it.

In [ ]:
def longest_unique_substring(text):
    """Length of the longest substring with no repeated character.

    O(n) time, O(min(n, alphabet)) space.
    """
    last_seen = {}
    left = 0
    best = 0
    best_text = ""
    for right, char in enumerate(text):
        # If we have seen it INSIDE the current window, jump left past it.
        if char in last_seen and last_seen[char] >= left:
            left = last_seen[char] + 1
        last_seen[char] = right
        if right - left + 1 > best:
            best = right - left + 1
            best_text = text[left:right + 1]
    return best, best_text


for sample in ("abcabcbb", "bbbbb", "pwwkew", "", "dvdf"):
    length, substring = longest_unique_substring(sample)
    print(f"  {sample!r:<12} -> {length}  {substring!r}")

print("\n🔴 Note `last_seen[char] >= left`. Without it, a character seen")
print("   BEFORE the window would wrongly drag `left` backwards - and left")
print("   must never move back, or the O(n) argument collapses.")
print("   'dvdf' is the case that catches this: the answer is 3, not 2.")

In [ ]:
def min_window_at_least(data, target):
    """Shortest contiguous run summing to >= target. Positive numbers only.

    The canonical grow/shrink window. O(n) time, O(1) space.
    """
    left = 0
    total = 0
    best = float("inf")
    best_range = None

    for right, value in enumerate(data):
        total += value                          # grow
        while total >= target:                  # shrink while still valid
            if right - left + 1 < best:
                best = right - left + 1
                best_range = (left, right)
            total -= data[left]
            left += 1

    return (0, None) if best == float("inf") else (best, best_range)


series = [2, 3, 1, 2, 4, 3]
for target in (7, 15, 4):
    length, span = min_window_at_least(series, target)
    detail = f"{series[span[0]:span[1] + 1]}" if span else "no such run"
    print(f"  target >= {target:<3} -> length {length}  {detail}")

print("\n🔴 'Positive numbers only' matters. With negatives, adding an")
print("   element can DECREASE the sum, so shrinking on a valid window is")
print("   no longer safe - you need prefix sums plus a different technique.")

---

# Pattern 3: Prefix sums

Precompute cumulative totals once, then answer any range query in O(1).

```
   data     [ 3,  1,  4,  1,  5 ]
   prefix [0, 3,  4,  8,  9, 14 ]      prefix[i] = sum of the first i items

   sum(data[1:4]) = prefix[4] - prefix[1] = 9 - 3 = 6
                    ^^^^^^^^^^^^^^^^^^^^ one subtraction, O(1)
```

| | Recompute each query | Prefix sums |
|---|---|---|
| Setup | none | O(n) once |
| Per query | O(n) | **O(1)** |
| q queries | O(q·n) | **O(n + q)** |

The leading `0` is not decoration — it removes the special case for ranges starting at index 0, and it is where most off-by-one bugs in this pattern come from.

**When to reach for it:** many range-sum queries over data that does not change; subarray-sum problems; and the counting trick below.

In [ ]:
import itertools


def build_prefix(data):
    """prefix[i] = sum of data[:i]. Note the leading 0."""
    return [0] + list(itertools.accumulate(data))


def range_sum(prefix, start, stop):
    """Sum of data[start:stop] in O(1)."""
    return prefix[stop] - prefix[start]


data = [3, 1, 4, 1, 5, 9, 2, 6]
prefix = build_prefix(data)
print("data  :", data)
print("prefix:", prefix)
print()
for start, stop in ((1, 4), (0, 8), (5, 6), (0, 0)):
    computed = range_sum(prefix, start, stop)
    actual = sum(data[start:stop])
    print(f"  data[{start}:{stop}] -> {computed:>3}   matches sum(): {computed == actual}")

print("\nq queries over n items: O(n + q) instead of O(q * n).")

In [ ]:
from collections import defaultdict


def count_subarrays_with_sum(data, target):
    """How many contiguous subarrays sum to exactly target?

    The prefix-sum + hash-map trick. O(n) time and space, and it works
    with NEGATIVE numbers - which the sliding window cannot.

    If prefix[j] - prefix[i] == target, then a subarray (i, j] qualifies.
    So for each j, count how many earlier prefixes equal prefix[j] - target.
    """
    seen = defaultdict(int)
    seen[0] = 1                       # the empty prefix
    running = 0
    count = 0
    for value in data:
        running += value
        count += seen[running - target]
        seen[running] += 1
    return count


def count_brute(data, target):
    total = 0
    for i in range(len(data)):
        running = 0
        for j in range(i, len(data)):
            running += data[j]
            if running == target:
                total += 1
    return total


cases = [
    ([1, 1, 1], 2),
    ([1, 2, 3], 3),
    ([1, -1, 0], 0),                  # negatives and zeros
    ([3, 4, 7, 2, -3, 1, 4, 2], 7),
]
for values, target in cases:
    fast = count_subarrays_with_sum(values, target)
    slow = count_brute(values, target)
    print(f"  {str(values):<28} target {target:>3} -> {fast}  "
          f"(brute force agrees: {fast == slow})")

print("\n🔴 This is the pattern to reach for when the sliding window fails")
print("   because of negative numbers. O(n) instead of O(n^2).")

---

# Pattern 4: In-place manipulation

"In place" means **O(1) extra space** — you may use a handful of variables, but not another copy of the data.

```
   reverse:      swap ends, walk inwards
   rotate by k:  reverse all, then reverse each part

        [1,2,3,4,5]  k=2
   ->   [5,4,3,2,1]        reverse everything
   ->   [4,5,3,2,1]        reverse the first k
   ->   [4,5,1,2,3]        reverse the rest
```

The triple-reversal rotation is worth memorising: it is a common interview question and the trick is not obvious.

🔴 **In Python, watch out for slicing.** `data[::-1]` and `data[k:] + data[:k]` are clear and correct — and both **allocate a new list**, so they are O(n) space. If the question says in-place, they do not qualify.

In [ ]:
def reverse_in_place(data, start=0, stop=None):
    """Reverse data[start:stop] using O(1) extra space."""
    stop = len(data) if stop is None else stop
    left, right = start, stop - 1
    while left < right:
        data[left], data[right] = data[right], data[left]
        left += 1
        right -= 1


def rotate_right(data, k):
    """Rotate right by k, in place. Three reversals, O(n) time, O(1) space."""
    n = len(data)
    if n == 0:
        return
    k %= n                              # 🔴 k may exceed n
    reverse_in_place(data)
    reverse_in_place(data, 0, k)
    reverse_in_place(data, k, n)


values = [1, 2, 3, 4, 5]
print("start      :", values)
rotate_right(values, 2)
print("rotate by 2:", values)
rotate_right(values, 8)                 # 8 % 5 == 3
print("rotate by 8:", values, "  (8 % 5 = 3)")

check = [1, 2, 3, 4, 5]
rotate_right(check, 0)
print("rotate by 0:", check)

print("\nthe slicing version is clearer and uses O(n) space:")
sliced = [1, 2, 3, 4, 5]
k = 2
print("   data[-k:] + data[:-k] ->", sliced[-k:] + sliced[:-k])
print("\n🔴 Both are correct. Only one satisfies 'in place, O(1) space'.")
print("   Say which you are giving them, and why.")

## The recognition table

Most array and string interview questions map onto one of these. Learning to **recognise** which is more valuable than memorising any single solution.

| The question says | Reach for | Cost |
|---|---|---|
| sorted, find a **pair** | two pointers, converging | O(n) |
| palindrome, symmetry | two pointers, converging | O(n) |
| remove/filter **in place** | fast/slow read-write | O(n), O(1) space |
| contiguous, **fixed** length k | fixed sliding window | O(n) |
| longest/shortest contiguous with a condition | variable sliding window | O(n) |
| many **range sum** queries | prefix sums | O(n + q) |
| subarray sums **with negatives** | prefix sums + hash map | O(n) |
| find a pair/triple, unsorted | hash map (**15.6**) | O(n) |
| kth largest | heap (**15.8**) | O(n log k) |

### The one-question triage

> **Is the input sorted?** If yes, two pointers or binary search (**15.11**) are almost certainly intended. If no, ask whether sorting first is allowed — O(n log n) plus an O(n) scan beats O(n²), and "can I sort it?" is a strong question to ask out loud.

## Interview questions

**1. Two Sum (unsorted).** Return indices of two numbers adding to a target.
> One pass with a hash map: for each `x`, check whether `target - x` has been seen. O(n) time, O(n) space. Say the brute force is O(n²) and that sorting would allow two pointers at O(n log n) — but loses the original indices.

**2. Best time to buy and sell stock.** Maximum profit from one buy and one later sell.
> Track the minimum so far and the best profit so far in one pass. O(n) time, O(1) space. It is a sliding window in disguise.

**3. Container with most water.** Two lines forming the largest area.
> Converging two pointers, always moving the **shorter** side — moving the taller one can never help, since width shrinks and height is capped by the shorter. O(n).

**4. Longest substring without repeating characters.** *(implemented above)*
> Variable window plus a map of last positions. The `>= left` guard is the whole difficulty.

**5. Product of array except self, without division.**
> Two passes of prefix products — left-to-right, then right-to-left. O(n) time, O(1) extra space if the output array does not count.

**6. Merge two sorted arrays in place.**
> Fill from the **back**, so you never overwrite unread data. Filling forwards is the classic wrong answer.

**7. Move all zeros to the end, preserving order.**
> Fast/slow read-write pointer, then fill the tail. O(n), O(1) space.

**8. Maximum sum subarray (Kadane's algorithm).**
> At each element decide: extend the current run, or start fresh. O(n). This is also the simplest possible dynamic program — see **15.13**.

**9. Trapping rain water.**
> Two pointers tracking the max from each side. O(n) time, O(1) space. The prefix-array version is O(n) space and easier to explain — offer that first.

**10. Why is a variable-size sliding window O(n) and not O(n²)?**
> `left` only ever advances, so across the whole run each index is added once and removed once — at most 2n moves, despite the nested `while`.

In [ ]:
# Two of the above, since they are asked constantly.


def two_sum(numbers, target):
    """Unsorted input. One pass, hash map. O(n) time and space."""
    seen = {}
    for index, value in enumerate(numbers):
        complement = target - value
        if complement in seen:              # O(1) - 15.2
            return (seen[complement], index)
        seen[value] = index
    return None


def max_subarray(data):
    """Kadane. At each step: extend the run, or start again here."""
    if not data:
        return 0, None
    best = current = data[0]
    best_start = best_end = start = 0
    for i in range(1, len(data)):
        if current < 0:                     # the run so far is a liability
            current = data[i]
            start = i
        else:
            current += data[i]
        if current > best:
            best, best_start, best_end = current, start, i
    return best, (best_start, best_end)


print("two_sum([2,7,11,15], 9)  ->", two_sum([2, 7, 11, 15], 9))
print("two_sum([3,2,4], 6)      ->", two_sum([3, 2, 4], 6))
print("two_sum([1,2], 50)       ->", two_sum([1, 2], 50))

print()
for series in ([-2, 1, -3, 4, -1, 2, 1, -5, 4], [-3, -1, -2], [5]):
    total, span = max_subarray(series)
    print(f"  max_subarray({str(series):<32}) = {total:>3}  "
          f"from {series[span[0]:span[1] + 1]}")

print("\n🔴 The all-negative case is the one people get wrong: the answer is")
print("   the largest single element, not 0. Only return 0 if the empty")
print("   subarray is explicitly allowed - ask.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Using a sliding window on data containing negatives.** Adding an element can shrink the sum, so the shrink step is unsound. Use prefix sums + a hash map.
2. 🔴 **Letting `left` move backwards in a variable window.** The O(n) argument depends on it only ever advancing.
3. 🔴 **Calling a slicing solution 'in place'.** `data[::-1]` allocates a new list - that is O(n) space.
4. **Off-by-one in prefix sums.** Keep the leading `0` and the rule `sum(data[i:j]) == prefix[j] - prefix[i]`.
5. **Forgetting `k %= n` when rotating.** A rotation larger than the array otherwise indexes out of range.
6. **Returning 0 for an all-negative maximum subarray.** The answer is the largest element unless the empty subarray is allowed.
7. **Merging sorted arrays front-to-back in place.** You overwrite values you have not read. Fill from the back.
8. **Assuming two pointers work on unsorted data.** The technique depends on order; otherwise use a hash map.
9. **Not asking whether you may sort or modify the input.** It changes which solutions are available.

## Best Practices

- Ask whether the input is sorted, may be sorted, or may be modified - before designing.
- State time *and* space complexity for the approach you choose (**15.1**).
- Write the clear O(n)-space version first, then optimise to O(1) if asked.
- Test the empty input, a single element, and all-negative cases - they are where the bugs are.
- Prefer `enumerate` over manual index bookkeeping.
- Name your pointers for their role - `left`/`right`, `read`/`write` - not `i`/`j`.
- Explain the amortised argument when a `while` sits inside a `for`; the interviewer is waiting for it.
- Reach for prefix sums whenever the same range is summed more than once.

## Practice Exercises

Try these before moving on.

1. Implement 'move zeros to the end' in place, preserving the order of the rest. Then prove it is O(n) with O(1) extra space.
2. Implement 'product of array except self' without division, in two passes.
3. 🔴 Implement 'merge two sorted arrays in place' filling forwards, and find an input where it corrupts the data. Then fix it by filling from the back.
4. Extend `min_window_at_least` to return the actual subarray, and test it against an input where no window qualifies.
5. Implement 'trapping rain water' twice - once with prefix max arrays (O(n) space) and once with two pointers (O(1) space). Verify they agree on random inputs.
6. Implement 'container with most water' and explain, in one sentence, why always moving the shorter side is safe.
7. Take `count_subarrays_with_sum` and adapt it to count subarrays whose sum is **divisible by k**. What changes about the hash-map key?
8. 🔴 Write a brute-force checker and a random-input generator, and use them to verify three of the functions in this notebook. This is how you build confidence in an algorithm you cannot fully prove.